# 01 Appliance 40 ms Contract

Rebuilds canonical data from the raw interleaved appliance CSV and proves the corrected timing contract.

In [2]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_DIR = Path(os.environ.get("AMPERE_DATA_DIR", "data/raw"))
RUN_MODE = os.environ.get("AMPERE_RUN_MODE", "smoke")
SEEDS = [42, 123, 2026]
OUTPUT_DIR = Path(os.environ.get("AMPERE_OUTPUT_DIR", "runs"))

try:
    display
except NameError:
    display = print

print("ROOT= <repo root>")
print("DATA_DIR=", DATA_DIR)
print("RUN_MODE=", RUN_MODE)
print("OUTPUT_DIR=", OUTPUT_DIR)

ROOT= <repo root>
DATA_DIR= data\raw
RUN_MODE= smoke
OUTPUT_DIR= runs


In [3]:
from ampere_public.publication import build_public_canonical_outputs

manifest = build_public_canonical_outputs(DATA_DIR, OUTPUT_DIR / "processed")
appliance = manifest["datasets"]["appliance_8ch"]
display(appliance)

assert appliance["raw_value_count"] == 360000
assert appliance["rows"] == 45000
assert appliance["branch_count"] == 8
assert appliance["duration_s"] == 1800
assert appliance["dt_s"] == 0.04
assert appliance["dwell_samples"] == 1
assert appliance["dwell_s_effective"] == 0.04
assert appliance["scan_cycle_s"] == 0.32
assert appliance["time_axis_confidence"] == "time_major_interleaved_dense_truth_40ms"
print("Corrected appliance timing contract verified.")

{'dataset_id': 'appliance_8ch', 'source_type': 'component_circuit', 'source_file': 'data/raw/combined_output.csv', 'raw_value_count': 360000, 'rows': 45000, 'wide_shape': [45000, 49], 'branch_count': 8, 'branch_names': ['Branch01_Ceiling_Fan', 'Branch02_Tubelight', 'Branch03_Electric_Kettle', 'Branch04_Electric_Geyser', 'Branch05_Water_Pump', 'Branch06_Refrigerator', 'Branch07_Rice_Cooker', 'Branch08_Microwave_Oven'], 'samples_per_branch': 45000, 'expected_samples_per_branch': 45000, 'duration_s': 1800, 'dt_s': 0.04, 'time_axis_confidence': 'time_major_interleaved_dense_truth_40ms', 'power_scale': 501530.0, 'dwell_s_requested': 0.04, 'dwell_samples': 1, 'dwell_s_effective': 0.04, 'scan_cycle_s': 0.32, 'dwell_policy': 'native 40 ms physical scan mask: one selected branch per 0.04 s row', 'power_representation': 'raw_signed_power'}
Corrected appliance timing contract verified.


In [4]:
import pandas as pd

wide_path = OUTPUT_DIR / "processed" / "appliance_8ch_wide.parquet"
wide = pd.read_parquet(wide_path)
preview = wide[["time_index", "time_s", "selected_branch", "dwell_id", "scan_cycle_id", "dt_s", "dwell_s_effective"]].head(12)
display(preview)

assert list(wide["selected_branch"].head(8)) == [1, 2, 3, 4, 5, 6, 7, 8]
assert list(wide["time_s"].head(9).round(2)) == [0.0, 0.04, 0.08, 0.12, 0.16, 0.20, 0.24, 0.28, 0.32]
print("Native 40 ms round-robin scan mask verified.")

    time_index  time_s  selected_branch  ...  scan_cycle_id  dt_s  dwell_s_effective
0            0    0.00                1  ...              0  0.04               0.04
1            1    0.04                2  ...              0  0.04               0.04
2            2    0.08                3  ...              0  0.04               0.04
3            3    0.12                4  ...              0  0.04               0.04
4            4    0.16                5  ...              0  0.04               0.04
5            5    0.20                6  ...              0  0.04               0.04
6            6    0.24                7  ...              0  0.04               0.04
7            7    0.28                8  ...              0  0.04               0.04
8            8    0.32                1  ...              1  0.04               0.04
9            9    0.36                2  ...              1  0.04               0.04
10          10    0.40                3  ...              1  0.04